# AI-Based Predictive Maintenance for Mining Equipment
**Final code submission | Group 9 | Saskatchewan Polytechnic**  
Ganesh Mishra | Lakshya Sharma | Sagar Patel  
Supervisor Dr. Himanshu Patel | November 2026

This notebook explains the implemented workflow, shows archived experiments and executes available saved models for Pump, Haul Truck, Belt Conveyor, Dozer, Excavator, Wheel Loader, Drill Rig and Motor Grader.

Equipment names describe mining **component application mappings**. The data come from laboratory rigs, event cameras, infrared recordings, real gear-tooth photographs and synthetic gear renders. The implemented task is current fault classification. Field reliability, future failure time and remaining useful life have not been established.

## Run the submission
1. Extract Group9_Capstone_Code.zip and install requirements.txt in a virtual environment.
2. Open this notebook from the extracted Notebook folder and choose that Python kernel.
3. For live inference, supply the existing Raw_Data, Images and Processed_Data folders, including the operator-controlled model trust registry.
4. Run all cells. Saved-result sections use the included JSON evidence. Missing local prerequisites are explicitly reported.

**Run All does not fit or replace models.** The complete preprocessing, training, evaluation and application implementation is included in Notebook/predictive_maintenance.py. The notebook calls that same code to keep results consistent with the CLI and web application.

Source revision 5ffc2638f91d14229928d1c3e825cef03b76d0b3. This submission copy preserves the two original notebooks in the repository.

**Saved output provenance.** The displayed outputs record verification in the configured local project environment. Run All refreshes availability and inference status on the current computer. The ZIP excludes local model checkpoints, trust registries and datasets.

## 1 Import the implementation
The setup locates the extracted package or repository using its Notebook folder. All paths are relative to that project. CAPSTONE_PROJECT_DIR can point to a separately configured repository when starting Jupyter elsewhere.

The Python module provides the actual preprocessing, checkpoint validation, model selection and prediction routes. The submission includes its full source.

In [1]:
from pathlib import Path
import os
import sys
import json
import io
import inspect
import contextlib
from tempfile import TemporaryDirectory
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

search_roots = []
if os.environ.get("CAPSTONE_PROJECT_DIR"):
    search_roots.append(Path(os.environ["CAPSTONE_PROJECT_DIR"]).expanduser())
for folder in (Path.cwd(), *Path.cwd().parents):
    search_roots.extend([folder, folder / "Group9_Capstone_Code"])
PROJECT_ROOT = next(
    (folder for folder in search_roots
     if (folder / "Notebook" / "predictive_maintenance.py").is_file()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Extract the code ZIP, open its Notebook folder, or set CAPSTONE_PROJECT_DIR.")
sys.path.insert(0, str(PROJECT_ROOT / "Notebook"))
import predictive_maintenance as app

equipment_names = {
    "pump": "Pump", "haul_truck": "Haul Truck", "belt_conveyor": "Belt Conveyor",
    "dozer": "Dozer", "excavator": "Excavator", "wheel_loader": "Wheel Loader",
    "drill_rig": "Drill Rig", "grader": "Motor Grader",
}
print("Project module imported. Eight application mappings loaded.")

Project module imported. Eight application mappings loaded.


## 2 Review the targets and checkpoint prerequisites
Pump and Haul Truck compare healthy, inner-race, outer-race and ball faults on separate rigs. CORA is one rig mapped to four applications. Dozer reuses Belt Conveyor data, while Excavator and Wheel Loader share healthy and bearing runs. Drill Rig compares gearbox vibration with CAD renders; root crack and rendered root breakage require qualification.

Motor Grader has two sensor tasks. Aalto healthy-versus-damaged gear runs support the broad comparison with gear-tooth photographs. The original JUST four-class bearing route remains separate.

The availability table shows whether files are present and registered. Actual inference is tested in section 8. A registration alone is not a verified digest match; the loader checks model bytes against SHA-256 at loading. Seven ResNet18 checkpoints use weights-only loading. Joblib checkpoints also require trusted hashes.

In [2]:
mapping_rows = []
for key, name in equipment_names.items():
    config = app.MACHINES[key]
    mapping_rows.append({
        "Mapping": name, "Component": config.component,
        "Configured classes": ", ".join(config.labels)
    })
display(pd.DataFrame(mapping_rows).style.hide(axis="index"))
display(pd.DataFrame(app.checkpoint_status()).style.hide(axis="index"))
print("Additional Motor Grader gear labels:", ", ".join(app.GRADER_GEAR_LABELS))

Mapping,Component,Configured classes
Pump,support bearing,"healthy, inner, outer, ball"
Haul Truck,wheel-motor bearing,"healthy, inner, outer, ball"
Belt Conveyor,geared drive,"healthy, wear25, wear50, wear75"
Dozer,final drive,"healthy, wear25, wear50, wear75"
Excavator,drive train,"healthy, bearing, unbalance, misalignment"
Wheel Loader,electric drive,"healthy, bearing, half_broken_bar, broken_bar"
Drill Rig,rotary-head reduction gear,"healthy, gear_wear, gear_pitting, tooth_crack"
Motor Grader,circle-drive slewing bearing,"healthy, inner_ring, outer_ring, rolling_element"


machine,route,checkpoint,present,registered,route_check
Pump,sensor,pump_sensor.joblib,True,True,True
Pump,image,pump_images.pt,True,True,True
Haul Truck,sensor,haul_truck_sensor.joblib,True,True,True
Haul Truck,image,haul_truck_images.pt,True,True,True
Belt Conveyor,sensor,belt_conveyor_sensor.joblib,True,True,True
Belt Conveyor,image,belt_conveyor_images.pt,True,True,True
Excavator,sensor,excavator_sensor.joblib,True,True,True
Excavator,image,excavator_images.pt,True,True,True
Dozer,sensor,dozer_sensor.joblib,True,True,True
Dozer,image,dozer_images.pt,True,True,True


Additional Motor Grader gear labels: healthy, tooth_damage


## 3 Read and preprocess real sensor recordings
The example uses the existing Excavator CORA task. Its reader validates numeric channels, creates one-second windows and extracts time and frequency features. Recording ID, class, speed and window order remain metadata; labels are not feature inputs.

RMS, standard deviation, crest factor, skewness and kurtosis summarize the waveform. Spectral bands, dominant frequency, centroid and entropy describe frequency content. Shaft-order features normalize frequency relative to rotation where source metadata permits. The UODS acceleration mean is excluded because a source-specific offset could reveal the label.

The same source-specific preprocessing is reused at inference. No synthetic data are substituted when files are missing.

In [3]:
sensor_table = None
feature_sets = None
config = app.MACHINES["excavator"]
required_cora = [
    app.RAW / "cora_vibration" / f"{label}_{speed}_S_Vx.csv"
    for label in ("H", "BD", "U", "M")
    for speed in ("F5", "F15", "F50", "F60")
]
missing = [path.name for path in required_cora if not path.is_file()]
if missing:
    print("Preprocessing example unavailable. Missing CORA files:", len(missing))
else:
    sensor_table, feature_sets = config.load_sensor()
    print(f"{len(sensor_table):,} windows from {sensor_table['unit'].nunique()} recordings.")
    print("Feature sets:", {name: len(columns) for name, columns in feature_sets.items()})
    display(sensor_table.head(5))
    display(sensor_table.groupby(["class", "speed"])["unit"].nunique().unstack())

4,800 windows from 16 recordings.
Feature sets: {'vibration': 13}


,unit,speed,class,window,log_rms,kurtosis,skewness,crest_factor,peak_order,log_ratio_2x_1x,order_power_0.5_1.5,order_power_1.5_2.5,order_power_2.5_3.5,order_power_3.5_5.5,order_power_5.5_8.5,order_power_8.5_12.5,order_power_12.5_20.5
0,H_F5_S,F5,healthy,0,-2.465438,3.895056,0.701620,4.237327,12.6,1.019994,0.013531,0.037525,0.037162,0.105805,0.108175,0.140393,0.557409
1,H_F5_S,F5,healthy,1,-2.435367,4.309689,0.810455,4.963230,15.0,-0.150602,0.086317,0.074249,0.023325,0.098933,0.183262,0.134534,0.399381
2,H_F5_S,F5,healthy,2,-2.461196,4.277627,0.786006,5.383057,9.0,0.162279,0.047090,0.055387,0.124455,0.030021,0.156568,0.262394,0.324084
3,H_F5_S,F5,healthy,3,-2.418389,4.285097,0.915171,4.571230,4.2,-0.009847,0.029097,0.028812,0.111938,0.194581,0.136610,0.218724,0.280239
4,H_F5_S,F5,healthy,4,-2.443105,4.195565,0.848105,5.205089,12.6,-0.856115,0.098764,0.041956,0.020207,0.104022,0.176271,0.126038,0.432742


speed,F15,F5,F50,F60
class,,,,
bearing,1,1,1,1
healthy,1,1,1,1
misalignment,1,1,1,1
unbalance,1,1,1,1


## 4 Verify primary and time splits
**Primary grouped holdout** keeps every observation in an outer test group outside training. CORA holds out speeds, Haul Truck sensor tests hold out bearing identities, and Aalto holds out gear pairs. A condition holdout can still reuse a specimen at another condition. Inner training-group validation selects models, features or epochs.

**Secondary time split** trains on the first 60 percent of each recording, drops a 10 percent gap and tests the last 30 percent. It measures later portions of known recordings. It does not establish unseen-machine or unseen-speed transfer.

This cell verifies recording separation in the real CORA outer folds and prints the actual time-split window counts.

In [4]:
if sensor_table is None:
    print("Split demonstration unavailable because the sensor example is missing.")
else:
    folds = app.fair_test_folds(sensor_table[config.sensor_group], config.sensor_held_out)
    rows = []
    for held_out, test_mask in zip(config.sensor_held_out, folds):
        train_units = set(sensor_table.loc[~test_mask, "unit"])
        test_units = set(sensor_table.loc[test_mask, "unit"])
        assert not train_units.intersection(test_units)
        rows.append({
            "Held-out group": str(held_out), "Train recordings": len(train_units),
            "Test recordings": len(test_units), "Recording overlap": 0
        })
    display(pd.DataFrame(rows).style.hide(axis="index"))
    keep, test, split_units = app.time_split(
        sensor_table["unit"].to_numpy(), sensor_table["window"].to_numpy()
    )
    print(f"Time split: {(~test).sum()} training windows, {test.sum()} test windows, "
          f"{(~keep).sum()} discarded gap windows.")

Held-out group,Train recordings,Test recordings,Recording overlap
['F5'],12,4,0
['F15'],12,4,0
['F50'],12,4,0
['F60'],12,4,0


Time split: 2880 training windows, 1440 test windows, 480 discarded gap windows.


## 5 Explain training and scoring without retraining
Five sensor families are evaluated: random forest, extra trees, histogram gradient boosting, RBF SVM and logistic regression. Standardization stays inside SVM and logistic pipelines and is fitted on training rows. Grouped validation macro F1 selects the estimator and feature set. The final deployed estimator can vary by mapping; the application reports the actual model.

Seven visual classifiers fine-tune ImageNet ResNet18 with a replacement output layer. AdamW uses learning rate 0.0003, weight decay 0.0001 and batches of 32; validation selects training duration. Image normalization and augmentation follow the machine configuration. Motor Grader photographs use frozen ResNet18 features and balanced logistic regression with C 0.1 and crop margin 0.15, fixed in advance.

The class with the largest mean probability across a recording's windows or frames becomes its prediction. Accuracy is correct units divided by all units. Balanced accuracy is mean class recall. Macro F1 averages class F1 equally. Evaluation units differ among recordings, runs, rendered gears and tooth crops.

The next cell reads the actual candidate settings and displays the source of the aggregation function. It does not fit a model.

In [5]:
candidate_settings = []
for name, factory in app.sensor_candidates().items():
    estimator = factory()
    selected_parameters = {
        key: value for key, value in estimator.get_params().items()
        if key in {"n_estimators", "min_samples_leaf", "class_weight", "max_iter",
                   "learning_rate", "random_state"}
        or key.endswith(("__C", "__gamma", "__class_weight"))
    }
    candidate_settings.append({"Candidate": name, "Settings": str(selected_parameters)})
display(pd.DataFrame(candidate_settings).style.hide(axis="index"))
print(inspect.getsource(app.predict_recordings))

Candidate,Settings
Random forest,"{'class_weight': 'balanced_subsample', 'min_samples_leaf': 2, 'n_estimators': 250, 'random_state': 42}"
Extra trees,"{'class_weight': 'balanced', 'min_samples_leaf': 2, 'n_estimators': 300, 'random_state': 42}"
Gradient boosting,"{'class_weight': 'balanced', 'learning_rate': 0.05, 'max_iter': 200, 'min_samples_leaf': 20, 'random_state': 42}"
SVM (RBF),"{'svc__C': 10.0, 'svc__class_weight': 'balanced', 'svc__gamma': 'scale'}"
Logistic regression,"{'logisticregression__C': 1.0, 'logisticregression__class_weight': 'balanced'}"


def predict_recordings(probabilities, classes, y, units):
    """One prediction per test unit (recording, run or gear): average the class probabilities of its rows."""
    actual, predicted = [], []
    for unit in sorted(set(units)):
        rows = units == unit
        if len(set(y[rows])) != 1:
            raise ValueError(f"Mixed labels within {unit}")
        actual.append(y[rows][0])
        predicted.append(classes[int(np.argmax(probabilities[rows].mean(axis=0)))])
    return actual, predicted



## 6 Inspect archived held-out performance
These are the genuine saved experimental scores. They are separate from deployment-file availability and are not a fresh evaluation run.

Motor Grader Aalto is the matched gear task, while JUST is the legacy bearing task. The photo model scores tooth crops within held-out source groups; Aalto scores gear-pair runs. Their broad healthy/damaged target is comparable with limitations, but the samples are independent and are not fused.

Weak whole-speed CORA results remain visible. Dozer duplicates Belt Conveyor evidence. Drill Rig visual scores concern synthetic renders. High accuracy on imbalanced gear photos must be read alongside balanced accuracy and damaged-tooth recall.

In [6]:
def read_result(path):
    return json.loads(path.read_text(encoding="utf-8"))

primary = {}
records = []
for key, name in equipment_names.items():
    primary[key] = {}
    for modality, filename, block_key in [
        ("Sensor", "sensor.json", "score"), ("Visual", "images.json", "unit_level")
    ]:
        saved = read_result(app.RESULTS / key / filename)
        block = saved[block_key]
        primary[key][modality] = (saved["classes"], block)
        records.append({
            "Mapping": name, "Modality": modality,
            "Task": "Legacy bearing" if key == "grader" and modality == "Sensor" else "Component condition",
            "Scored units": block["test_units"], "Accuracy (%)": block["accuracy"] * 100,
            "Balanced accuracy (%)": block["balanced_accuracy"] * 100,
            "Macro F1 (%)": block["macro_f1"] * 100,
        })
aalto = read_result(app.RESULTS / "experiments" / "grader_gear" / "aalto_sensor.json")
records.append({
    "Mapping": "Motor Grader", "Modality": "Sensor", "Task": "Aalto gear",
    "Scored units": aalto["score"]["test_units"],
    "Accuracy (%)": aalto["score"]["accuracy"] * 100,
    "Balanced accuracy (%)": aalto["score"]["balanced_accuracy"] * 100,
    "Macro F1 (%)": aalto["score"]["macro_f1"] * 100,
})
primary_table = pd.DataFrame(records)
display(primary_table.round(2).style.hide(axis="index"))

Mapping,Modality,Task,Scored units,Accuracy (%),Balanced accuracy (%),Macro F1 (%)
Pump,Sensor,Component condition,24,95.830000,95.830000,95.800000
Pump,Visual,Component condition,24,100.000000,100.000000,100.000000
Haul Truck,Sensor,Component condition,30,80.000000,73.330000,74.100000
Haul Truck,Visual,Component condition,12,58.330000,58.330000,56.550000
Belt Conveyor,Sensor,Component condition,16,43.750000,43.750000,35.000000
Belt Conveyor,Visual,Component condition,16,50.000000,50.000000,43.180000
Dozer,Sensor,Component condition,16,43.750000,43.750000,35.000000
Dozer,Visual,Component condition,16,56.250000,56.250000,52.080000
Excavator,Sensor,Component condition,16,25.000000,25.000000,18.330000
Excavator,Visual,Component condition,16,25.000000,25.000000,21.670000


### Explore the saved confusion matrices
Rows are actual labels and columns are predicted labels. Change the dropdown to inspect any mapping. Motor Grader shows Aalto gear sensor results alongside tooth photographs, with the legacy bearing score retained in the previous table.

Counts are evaluation units rather than independent machines. Good teeth greatly outnumber damaged teeth in the Grader photos. Displaying a saved matrix is distinct from running a new prediction.

In [7]:
def show_confusion_matrices(machine):
    sensor_labels, sensor_score = (
        (aalto["classes"], aalto["score"]) if machine == "grader"
        else primary[machine]["Sensor"]
    )
    visual_labels, visual_score = primary[machine]["Visual"]
    figure, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for axis, labels, block, title in [
        (axes[0], sensor_labels, sensor_score, "Sensor"),
        (axes[1], visual_labels, visual_score, "Visual")
    ]:
        matrix = np.asarray(block["confusion_matrix"])
        axis.imshow(matrix, cmap="Blues")
        axis.set_xticks(range(len(labels)), labels, rotation=30, ha="right")
        axis.set_yticks(range(len(labels)), labels)
        axis.set_xlabel("Predicted")
        axis.set_ylabel("Actual")
        axis.set_title(f"{title}: {block['test_units']:,} units, accuracy {block['accuracy']:.2%}")
        for (row, column), count in np.ndenumerate(matrix):
            axis.text(column, row, str(count), ha="center", va="center",
                      color="white" if count > matrix.max() / 2 else "black")
    figure.suptitle(equipment_names[machine])
    figure.tight_layout()
    plt.show()

matrix_choice = widgets.Dropdown(
    options=[(name, key) for key, name in equipment_names.items()],
    value="pump", description="Mapping"
)
matrix_output = widgets.interactive_output(show_confusion_matrices, {"machine": matrix_choice})
display(matrix_choice, matrix_output)

Dropdown(description='Mapping', options=(('Pump', 'pump'), ('Haul Truck', 'haul_truck'), ('Belt Conveyor', 'be…

Output()

## 7 Inspect the saved time-split results
Five mappings have secondary time-split experiments. They test later segments of recordings used in training. Pump, Drill Rig and Motor Grader have no saved secondary time-split experiment.

Haul Truck image time splits score 24 recording segments, while its primary image test scores 12 held-out recordings. Sample pools and fitting settings can differ between protocols. These scores therefore answer different questions and cannot establish the isolated effect of changing the split.

In [8]:
time_rows = []
for key in ["haul_truck", "belt_conveyor", "dozer", "excavator", "wheel_loader"]:
    for modality, filename, block_key in [
        ("Sensor", "sensor_timesplit.json", "score"),
        ("Visual", "images_timesplit.json", "unit_level")
    ]:
        block = read_result(app.RESULTS / key / filename)[block_key]
        time_rows.append({
            "Mapping": equipment_names[key], "Modality": modality,
            "Scored segments": block["test_units"], "Accuracy (%)": block["accuracy"] * 100,
            "Balanced accuracy (%)": block["balanced_accuracy"] * 100,
            "Macro F1 (%)": block["macro_f1"] * 100,
        })
time_table = pd.DataFrame(time_rows)
display(time_table.round(2).style.hide(axis="index"))
comparison = primary_table[
    primary_table["Mapping"].isin(time_table["Mapping"])
][["Mapping", "Modality", "Accuracy (%)"]].merge(
    time_table[["Mapping", "Modality", "Accuracy (%)"]],
    on=["Mapping", "Modality"], suffixes=(" held out", " time split")
)
display(comparison.round(2).style.hide(axis="index"))

Mapping,Modality,Scored segments,Accuracy (%),Balanced accuracy (%),Macro F1 (%)
Haul Truck,Sensor,30,100.000000,100.000000,100.000000
Haul Truck,Visual,24,100.000000,100.000000,100.000000
Belt Conveyor,Sensor,16,100.000000,100.000000,100.000000
Belt Conveyor,Visual,16,100.000000,100.000000,100.000000
Dozer,Sensor,16,100.000000,100.000000,100.000000
Dozer,Visual,16,93.750000,93.750000,93.650000
Excavator,Sensor,16,93.750000,93.750000,93.650000
Excavator,Visual,16,100.000000,100.000000,100.000000
Wheel Loader,Sensor,16,93.750000,93.750000,93.650000
Wheel Loader,Visual,16,100.000000,100.000000,100.000000


Mapping,Modality,Accuracy (%) held out,Accuracy (%) time split
Haul Truck,Sensor,80.000000,100.000000
Haul Truck,Visual,58.330000,100.000000
Belt Conveyor,Sensor,43.750000,100.000000
Belt Conveyor,Visual,50.000000,100.000000
Dozer,Sensor,43.750000,100.000000
Dozer,Visual,56.250000,93.750000
Excavator,Sensor,25.000000,93.750000
Excavator,Visual,25.000000,100.000000
Wheel Loader,Sensor,31.250000,93.750000
Wheel Loader,Visual,18.750000,100.000000


## 8 Execute real saved-model prediction routes
This cell performs inference with available local recordings and images. The existing checker writes its operational report under Processed_Data/deployment.

**PASS means the route executed successfully.** It can still predict an incorrect class. The mismatch table exposes such cases. Example inputs can belong to a deployment fit, so these checks do not establish held-out accuracy. Missing files remain explicitly unavailable.

An unsupported input is rejected. A CORA recording without a confirmed mapping remains ambiguous. No checkpoint is loaded by bypassing its trust validation.

In [9]:
with contextlib.redirect_stdout(io.StringIO()):
    live_verification = app.verify_routes()
live_rows = pd.DataFrame(live_verification["routes"])
summary = live_rows.groupby(["machine", "modality", "status"], dropna=False).size()
display(summary.rename("Checks").reset_index().style.hide(axis="index"))
display(pd.DataFrame(live_verification["input_checks"]).style.hide(axis="index"))
successful = live_rows[live_rows["status"] == "PASS"]
print(f"{len(successful)} of {len(live_rows)} inference checks executed successfully.")
if {"expected", "predicted"}.issubset(successful.columns):
    mismatches = successful[successful["expected"] != successful["predicted"]]
    print("Source-label disagreements among those sample predictions:", len(mismatches))
    display(mismatches[
        ["machine", "modality", "file", "expected", "predicted", "probability"]
    ].style.hide(axis="index"))
remaining = live_rows[live_rows["status"] != "PASS"]
if not remaining.empty:
    print("Unavailable or failed checks:")
    display(remaining.style.hide(axis="index"))

machine,modality,status,Checks
belt_conveyor,image,PASS,4
belt_conveyor,sensor,PASS,1
dozer,image,PASS,4
dozer,sensor,PASS,1
drill_rig,image,PASS,4
drill_rig,sensor,PASS,1
excavator,image,PASS,4
excavator,sensor,PASS,1
grader,gear sensor,PASS,1
grader,image,PASS,2


input,status,message,answer
unsupported file,PASS: rejected,README.md matches none of the known dataset formats; no prediction made,nan
CORA file without a mapping,PASS,nan,"{'file format': 'CORA rig vibration (Vx)', 'status': 'Ambiguous Equipment', 'candidates': ['Belt Conveyor', 'Excavator', 'Dozer', 'Wheel Loader']}"


39 of 39 inference checks executed successfully.
Source-label disagreements among those sample predictions: 2


machine,modality,file,expected,predicted,probability
drill_rig,image,T20ID10_gear_wear_00001_v0.png,gear_wear,healthy,0.438000
drill_rig,image,T20ID10_gear_pitting_00001_v0.png,gear_pitting,healthy,0.396000


## 9 Upload a sensor recording or component image
The interface calls the same prediction function as the CLI and local application. Use publisher channel names, units and acquisition rates. Compatible inputs include supported CSV/MAT recordings, event frames, infrared frames, indexed renders and confirmed tooth crops.

Choose a mapping only when source provenance is known. A choice confirms the laboratory application mapping; it does not validate a real mining machine. Whole-machine photographs do not establish internal faults. Motor Grader tooth inspection requires a crop of one tooth and the Motor Grader mapping.

Outputs show the condition, model probability and actual saved classifier. Probability is uncalibrated. Unknown, corrupted or incompatible inputs produce refusal or ambiguity rather than a guessed equipment identity.

In [10]:
file_upload = widgets.FileUpload(
    accept=".csv,.mat,.png,.jpg,.jpeg", multiple=False, description="Choose file"
)
mapping_choice = widgets.Dropdown(
    options=[("Automatic", None)] + [(name, key) for key, name in equipment_names.items()],
    description="Mapping", layout=widgets.Layout(width="380px")
)
run_prediction = widgets.Button(description="Predict condition", button_style="primary")
prediction_output = widgets.Output()

def display_prediction(_):
    prediction_output.clear_output(wait=True)
    with prediction_output:
        if not file_upload.value:
            print("Choose a sensor recording or supported component image.")
            return
        selected = file_upload.value[0]
        filename = str(selected["name"])
        if not app.upload_name_ok(filename):
            print("The filename is invalid.")
            return
        contents = selected["content"]
        if len(contents) > 300_000_000:
            print("File too large. The limit is 300 MB.")
            return
        run_prediction.disabled = True
        try:
            with TemporaryDirectory() as folder:
                path = Path(folder) / filename
                path.write_bytes(contents)
                answer = app.predict(path, mapping_choice.value)
            display(pd.DataFrame(
                [{"Field": key, "Value": str(value)} for key, value in answer.items()]
            ).style.hide(axis="index"))
        except Exception as problem:
            print("Prediction unavailable:", str(problem))
        finally:
            run_prediction.disabled = False

run_prediction.on_click(display_prediction)
display(widgets.VBox([mapping_choice, file_upload, run_prediction, prediction_output]))

## 10 Verify invalid input handling
These temporary files are deliberately invalid input-validation fixtures. They are not source measurements, training data or replacements for unavailable datasets. The real inference entry point must reject them.

In [11]:
invalid_cases = {
    "notes.txt": b"ordinary text",
    "bad.csv": b"unrecognized_column\nnot_a_number\n",
    "bad.png": b"not an image",
}
invalid_rows = []
with TemporaryDirectory() as folder:
    for filename, contents in invalid_cases.items():
        path = Path(folder) / filename
        path.write_bytes(contents)
        try:
            answer = app.predict(path)
        except (ValueError, FileNotFoundError) as problem:
            invalid_rows.append({"Input": filename, "Result": "Rejected", "Reason": str(problem)})
        else:
            invalid_rows.append({"Input": filename, "Result": str(answer.get("status", "Unexpected acceptance"))})
display(pd.DataFrame(invalid_rows).style.hide(axis="index"))
assert all(row["Result"] == "Rejected" for row in invalid_rows)

Input,Result,Reason
notes.txt,Rejected,notes.txt matches none of the known dataset formats; no prediction made
bad.csv,Rejected,Unrecognized CSV schema
bad.png,Rejected,bad.png matches none of the known dataset formats; no prediction made


## 11 Find the complete implementation

| Stage | Functions in predictive_maintenance.py |
|---|---|
| Sensor ingestion | ottawa_recording, uods_recording, cora_rig, mcc5_recording, just_recording, aalto_run_features |
| Visual preparation | source-specific export functions and tooth_crops |
| Grouped and time splits | fair_test_folds, check_no_overlap, time_split |
| Sensor fitting | sensor_candidates, choose_model, sensor_fair_test |
| CNN fitting and aggregation | fit_resnet, image_probabilities, predict_recordings |
| Gear-tooth photographs | evaluate_tooth_photos, predict_grader_tooth |
| Checkpoint validation | trusted_hashes, load_trusted_joblib, verified_checkpoint_bytes, cached_image_model |
| Automatic routing | identify, sensor_features, score_checkpoint, predict |
| Application verification | upload_server, verify_routes, run_demo_check |

From the project root, these commands reuse the existing models:

    python Notebook/predictive_maintenance.py route-check
    python Notebook/predictive_maintenance.py demo-check
    python Notebook/predictive_maintenance.py predict recording.csv --machine grader --no-history
    python Notebook/predictive_maintenance.py serve

For deliberate research reproduction with the original datasets:

    python Notebook/predictive_maintenance.py pump sensor
    python Notebook/predictive_maintenance.py belt_conveyor sensor --time-split
    python Notebook/predictive_maintenance.py belt_conveyor images --time-split

The research commands fit models or write experiment files. Run All does not call them.

## Sources and scientific limitations
Dataset versions, attribution and licences are in Dataset_Manifest/manifest.md. Main sources are Ottawa bearing data (10.17632/v43hmbwxpm.2), UODS-VAFDC (10.17632/y2px5tg92h.1), XJTU-DV (10.5281/zenodo.17447083), CORA (10.34810/DATA2500), MCC5-THU (10.17632/p92gj2732w.2), MFGNet-Gear (10.7302/qrdj-n812), JUST (10.17632/hwg8v5j8t6.1), GearBox_adddingImg and Aalto (10.17632/fywnj597d8.2).

- All measurements are laboratory analogues or synthetic images rather than mining-fleet validation.
- Dozer reuses Belt Conveyor data. CORA mappings share a rig and some run IDs.
- Haul Truck ball faults are load-confounded. Pump recording IDs do not establish independent bearing identities.
- Drill Rig synthetic root breakage and sensor root crack require qualification.
- Motor Grader gear runs and tooth crops come from different gears and use different evaluation units.
- Class probabilities are uncalibrated. Current fault classification is distinct from remaining life or future failure prediction.
- Eight research archives remain missing in the verification environment. Live inference uses the available local inputs and trusted checkpoints.
- The submission ZIP excludes raw datasets, image exports, model binaries and the local trust registry. Its saved-results cells can run without those assets; live inference requires them.